# Tutorial: BIGFAM Quickstart

This notebook explains BIGFAM in the simplest possible way.

BIGFAM has **two main objectives**:

1. **Objective 1**
   Estimate phenotypic correlation for each `DOR`, then run a slope test, then partition the result into `V_G` and `V_S`.

2. **Objective 2**
   Estimate phenotypic correlation for each `relationship`, then estimate `V_X`.
   If sex information is available, you can also extend this to sex-specific `V_X`.


## What You Will Do

- Load the sample data in `data/test/`
- Run **Objective 1** with `group_by='DOR'`
- Run **Objective 2** with `group_by='relationship'`
- See which BIGFAM function belongs to which objective


In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Could not find repo root from the current working directory.')

repo_root = find_repo_root(Path.cwd())
src_path = repo_root / 'src'
if str(src_path) in sys.path:
    sys.path.remove(str(src_path))
sys.path.insert(0, str(src_path))

import bigfam
from bigfam import BIGFAM

assert str(Path(bigfam.__file__).resolve()).startswith(str(repo_root)), 'Not using the local BIGFAM package.'
print('Repo root:', repo_root)
print('Python executable:', sys.executable)
print('Using bigfam package at:', bigfam.__file__)


Repo root: /Users/jerry/Documents/JerryProject/3.Resource/src/BIGFAM-core
Python executable: /Users/jerry/miniconda/envs/bigfam/bin/python
Using bigfam package at: /Users/jerry/Documents/JerryProject/3.Resource/src/BIGFAM-core/src/bigfam/__init__.py


## Step 0: Load Sample Data

We will use the reproducible sample files in `data/test/`.
This is the same starting point for both objectives.


In [ ]:
data_dir = repo_root / 'data' / 'test'
model = BIGFAM(verbose=False)
model.load_data(
    pheno_file=data_dir / 'binary.tsv',
    cov_file=data_dir / 'covariate.tsv',
    rel_file=data_dir / 'relationship.tsv',
    trait_col='trait',
    id_col='iid',
)
print('Loaded pairs:', len(model.df_pairs))


[Load Data]
  - covariates: ['age', 'sex', 'ageXsex', 'age2Xsex']
  - 19,215 phenotypes, 3 DORs, 54 relationships
  - CONTINUOUS, left 35,032 pairs, 13,665 unique IDs after merging
Loaded pairs: 35032


## Objective 1: DOR -> Slope Test -> `V_G`, `V_S`

This is the first main use of BIGFAM.

The idea is simple:

1. Group relative pairs by `DOR`
2. Estimate phenotypic correlation for each `DOR`
3. Check the slope pattern across DOR values
4. Partition the result into genetic variance (`V_G`) and shared-environment variance (`V_S`)

For this objective, the key function is:
- `estimate_correlations(group_by='DOR')`


In [3]:
df_dor = model.estimate_correlations(
    group_by='DOR',
    # method='volsummary',
    # n_bootstrap=10,
    # min_pairs=100,
)
print(df_dor.to_string(index=False))

[Estimate Correlations]
  - group : DOR, method: robust
  - analyzed: 3 groups
 DOR      rho       se  n_asym  n_vols  stable note
   1 0.174220 0.006523   16789   10450    True     
   2 0.054193 0.009323    7059    4501    True     
   3 0.047405 0.017318    2425    1499    True     


In [4]:
slope_result = model.run_slope_test(method='known_var')
print('\nSlope test result:')
print(slope_result)


Slope test result:
{'significance': 'similar', 'slope': np.float64(1.2944067370329417), 'slope_se': np.float64(0.1853144207349921), 'slope_lower': np.float64(0.9311904723923572), 'slope_upper': np.float64(1.6576230016735263), 'intercept': np.float64(-1.2341553581880977), 'intercept_se': np.float64(0.20416009071508473), 'intercept_lower': np.float64(-1.6343091359896638), 'intercept_upper': np.float64(-0.8340015803865317)}


In [11]:

var_result = model.estimate_variance_components()

print('\nObjective 1 summary:')
print({
    'V_G': var_result['V_G'],
    'V_S': var_result['V_S'],
    'w': var_result['w'],
    'slope_test_significance': var_result['slope_test_significance'],
})



[Variance Components] Starting estimation (CV mode)

  [Step 1] Setup
    - Slope test result: similar
    - w_S search range: [1.43, 3.33]
    - Cross-validation: 10-fold × 10 repeats = 100 runs
    - Loss function: log-scale SSE (original)

  [Step 2] Running cross-validation

[Variance Components] Done ✓ (CV mode)
  - V_G: 0.0000 (95% CI: [0.0000, 0.0000])
  - V_S: 0.1705 (95% CI: [0.1699, 0.1714])
  - w_S: 1.43 (95% CI: [1.43, 1.43])
  - Number of estimates: 100

Objective 1 summary:
{'V_G': np.float64(1e-06), 'V_S': np.float64(0.17054940879795039), 'w': np.float64(0.7000000000000004), 'slope_test_significance': 'similar'}


## Objective 2: Relationship -> `V_X`

This is the second main use of BIGFAM.

Now we stop grouping by `DOR` and instead group by the actual relationship type.
That gives a more detailed correlation table, which is then used to estimate `V_X`.

The key point is:
- **Objective 1 uses `group_by='DOR'`**
- **Objective 2 uses `group_by='relationship'`**

For this objective, the key function is:
- `estimate_correlations(group_by='relationship')`


In [12]:
df_rel = model.estimate_correlations(
    group_by='relationship',
)
print(df_rel.head().to_string(index=False))


[Estimate Correlations]
  - group : relationship, method: robust
  - analyzed: 28 groups
                   relationship      rho       se  n_asym  n_vols  stable note
                daughter-father 0.092775 0.016128    1963    1745    True     
        daughter-father-brother 0.131750 0.034444     482     412    True     
         daughter-father-sister 0.058224 0.028899     637     478    True     
daughter-father-sister-daughter 0.121756 0.041047     255     198    True     
                daughter-mother 0.109342 0.014015    3159    2817    True     


/Users/jerry/miniconda/envs/bigfam/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log2
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [13]:

x_result = model.estimate_x_variance(
    regout_bin=['DOR'],
)
print('\nObjective 2 summary:')
print({k: v for k, v in x_result.items() if k != 'raw_df'})



[X Estimation] Starting estimation
  - Resample: 1000 iterations
  - Regout bin: ['DOR']
  - Alpha weight: 2.0
  - Weighted Loss: False
  - Meta λ (IVW): 0.3342


  Bootstrap: 100%|██████████| 1000/1000 [00:08<00:00, 112.77it/s]


[X Estimation] Done ✓
  - V_X: 0.0000 (95% CI: [0.0000, 0.0055])

Objective 2 summary:
{'V_X': np.float64(1e-06), 'V_X_lower': np.float64(1e-06), 'V_X_upper': np.float64(0.005474500417812663)}


## What About Sex-Specific `V_X`?

If your relationship-level result has a `sex_type` column, you can continue to:

- `estimate_sex_specific_x()`

That gives:
- `V_X_male`
- `V_X_female`
- `r` (cross-sex environmental correlation)

You can think of this as an extended version of Objective 2.


In [ ]:
sex_x_result = model.estimate_sex_specific_x(
    regout_bin=['DOR', 'sex_type'],
)
print('\nSex-specific Objective 2 summary:')
print({k: v for k, v in sex_x_result.items() if k != 'raw_df'})



[Sex-Specific X Estimation] Starting estimation
  - Resample: 1000 iterations
  - Regout bin: ['DOR', 'sex_type']
  - Alpha weight: 2.0
  - r Grid: 11 points
  - Weighted Loss: False
  - Meta λ (IVW): 0.3342


  Bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

  Bootstrap:  14%|█▍        | 140/1000 [00:29<03:04,  4.65it/s]